# Accrue Quickstart: Account Enrichment

Enrich a list of accounts with **structured data** using LLMs — in under 20 lines of code.

**What you'll learn:**
- Define **field specs** inline to extract structured data
- Load field specs from a **CSV** (ops-friendly)
- Chain **FunctionStep → LLMStep** for multi-step pipelines

> **Prerequisites:** `pip install accrue` and set `OPENAI_API_KEY` in your environment.

In [1]:
import pandas as pd

from accrue import LLMStep, Pipeline

pd.set_option("display.max_colwidth", 80)

companies = [
    {
        "company": "Stripe",
        "website": "stripe.com",
        "description": "Online payment processing for internet businesses",
    },
    {
        "company": "Notion",
        "website": "notion.so",
        "description": "All-in-one workspace for notes, docs, and project management",
    },
    {
        "company": "Figma",
        "website": "figma.com",
        "description": "Collaborative interface design tool for teams",
    },
    {
        "company": "Datadog",
        "website": "datadoghq.com",
        "description": "Cloud monitoring and security platform",
    },
    {
        "company": "Canva",
        "website": "canva.com",
        "description": "Online design and visual communication platform",
    },
    {
        "company": "Plaid",
        "website": "plaid.com",
        "description": "Financial data connectivity for fintech applications",
    },
    {
        "company": "Airtable",
        "website": "airtable.com",
        "description": "Low-code platform for building collaborative apps",
    },
    {
        "company": "Vercel",
        "website": "vercel.com",
        "description": "Frontend cloud platform for web developers",
    },
]

pipeline = Pipeline(
    [
        LLMStep(
            "enrich",
            fields={
                "industry": {
                    "prompt": "Classify the company's primary industry",
                    "enum": [
                        "Fintech",
                        "Developer Tools",
                        "Productivity",
                        "Design",
                        "Infrastructure",
                        "Other",
                    ],
                },
                "icp_fit": {
                    "prompt": "Rate fit as a B2B SaaS prospect for a sales intelligence platform",
                    "enum": ["Strong Fit", "Moderate Fit", "Weak Fit"],
                },
                "employee_estimate": {
                    "prompt": "Estimate the number of employees",
                    "type": "Number",
                },
            },
        )
    ]
)

result = await pipeline.run_async(companies)
pd.DataFrame(result.data)

/Users/matthew.house/Programming/lattice/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Pipeline: 100%|██████████| 1/1 [00:02<00:00,  2.90s/step, step=enrich]


,company,website,description,industry,icp_fit,employee_estimate
0,Stripe,stripe.com,Online payment processing for internet businesses,Fintech,Strong Fit,9000.0
1,Notion,notion.so,"All-in-one workspace for notes, docs, and project management",Productivity,Strong Fit,1500.0
2,Figma,figma.com,Collaborative interface design tool for teams,Design,Strong Fit,1500.0
3,Datadog,datadoghq.com,Cloud monitoring and security platform,Infrastructure,Strong Fit,3000.0
4,Canva,canva.com,Online design and visual communication platform,Design,Strong Fit,3000.0
5,Plaid,plaid.com,Financial data connectivity for fintech applications,Fintech,Strong Fit,2000.0
6,Airtable,airtable.com,Low-code platform for building collaborative apps,Productivity,Strong Fit,2000.0
7,Vercel,vercel.com,Frontend cloud platform for web developers,Developer Tools,Strong Fit,800.0


In [2]:
print(f"Accounts enriched: {len(result.data)}")
print(f"Success rate:      {result.success_rate:.0%}")
print(f"Total tokens:      {result.cost.total_tokens:,}")

Accounts enriched: 8
Success rate:      100%
Total tokens:      4,726


---

## Load Fields from CSV

Your ops team defines prompts in a spreadsheet. Engineers wire the pipeline.

`load_fields()` reads a CSV of field specs — same dict format as inline definitions above.

In [3]:
from accrue.data import load_fields

# Non-technical team members manage prompts in a spreadsheet
fields = load_fields("field_categories.csv", category="account_qualification")
print(fields)  # Same dict format as inline specs

pipeline = Pipeline([LLMStep("enrich", fields=fields)])
result = await pipeline.run_async(companies)
pd.DataFrame(result.data)

{'industry': {'prompt': "Classify the company's primary industry", 'type': 'String', 'enum': ['Fintech', 'Developer Tools', 'Productivity', 'Design', 'Infrastructure', 'Other']}, 'icp_fit': {'prompt': 'Rate fit as a B2B SaaS prospect for a sales intelligence platform', 'type': 'String', 'enum': ['Strong Fit', 'Moderate Fit', 'Weak Fit']}, 'employee_estimate': {'prompt': 'Estimate the number of employees', 'type': 'Number'}, 'segment': {'prompt': "Classify the company's market segment based on likely revenue and team size", 'type': 'String', 'enum': ['Enterprise', 'Mid-Market', 'SMB']}}


Pipeline: 100%|██████████| 1/1 [00:01<00:00,  1.34s/step, step=enrich]


,company,website,description,industry,icp_fit,employee_estimate,segment
0,Stripe,stripe.com,Online payment processing for internet businesses,Fintech,Moderate Fit,8000.0,Enterprise
1,Notion,notion.so,"All-in-one workspace for notes, docs, and project management",Productivity,Strong Fit,2000.0,Enterprise
2,Figma,figma.com,Collaborative interface design tool for teams,Design,Strong Fit,1500.0,Enterprise
3,Datadog,datadoghq.com,Cloud monitoring and security platform,Infrastructure,Strong Fit,3000.0,Enterprise
4,Canva,canva.com,Online design and visual communication platform,Design,Strong Fit,3000.0,Enterprise
5,Plaid,plaid.com,Financial data connectivity for fintech applications,Fintech,Strong Fit,1500.0,Enterprise
6,Airtable,airtable.com,Low-code platform for building collaborative apps,Developer Tools,Strong Fit,1500.0,Enterprise
7,Vercel,vercel.com,Frontend cloud platform for web developers,Developer Tools,Strong Fit,500.0,Mid-Market


In [4]:
from accrue import FunctionStep


def lookup_firmographics(ctx):
    """Simulate CRM/API lookup returning firmographic data.
    Replace with your real data source (Clearbit, ZoomInfo, etc.)."""
    return {
        "__firmographics": "Founded in SF. Series C. 500+ employees. ARR $50M+.",
    }


pipeline = Pipeline(
    [
        FunctionStep("lookup", fn=lookup_firmographics, fields=["__firmographics"]),
        LLMStep(
            "qualify",
            fields={
                "qualification_summary": "Write a one-sentence sales qualification summary using the firmographic data",
            },
            depends_on=["lookup"],
        ),
    ]
)

result = await pipeline.run_async(companies)
pd.DataFrame(result.data)[["company", "qualification_summary"]]

Pipeline: 100%|██████████| 2/2 [00:02<00:00,  1.08s/step, step=qualify] 


,company,qualification_summary
0,Stripe,Stripe is a well-established online payment processing company based in San ...
1,Notion,"Notion is a well-established, fast-growing company based in San Francisco wi..."
2,Figma,Figma is a San Francisco-based collaborative interface design tool company w...
3,Datadog,Datadog is a well-established cloud monitoring and security platform company...
4,Canva,Canva is a well-established online design and visual communication platform ...
5,Plaid,Plaid is a well-established fintech company based in San Francisco with over...
6,Airtable,"Airtable is a San Francisco-based low-code platform with over 500 employees,..."
7,Vercel,"Vercel is a well-established frontend cloud platform for web developers, hea..."


> **Note:** The `__firmographics` field is prefixed with `__`, so it's passed between steps but **filtered from the final output** automatically.

---

## Next Steps

See **[advanced_pipeline.ipynb](advanced_pipeline.ipynb)** to:

- Add **caching** to skip re-enrichment ($0 on reruns)
- Use **conditional steps** to only deep-analyze high-value accounts
- Switch **providers** (Anthropic, Google, Ollama) with one line
- Add **grounding** for real-time web data
- Wire up **lifecycle hooks** for observability